# 📖 Notebook 2: Privacy Impact Assessment (PIA)

A **Privacy Impact Assessment** is a structured process to evaluate the privacy risks of a new feature **before** it launches. At Microsoft, Google, and other large tech companies, no feature that handles personal data can ship without a completed PIA.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to build a PIA workflow with structured risk evaluation
- How to score privacy risks using a consistent framework
- How to document data flows (what data goes where)
- How to track PIA status through the review lifecycle
- Why PIAs exist and what happens when you skip them

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 08-enterprise/privacy-review
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
from datetime import datetime, timedelta

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "privacy_review",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 Why Do PIAs Exist?

Imagine you're building a new feature: **"Personalized Product Recommendations"**. Sounds harmless, right?

But think about what it needs:
- User's **browsing history** (every page they visited)
- User's **purchase history** (what they bought)
- User's **location** (to recommend local products)
- Maybe even **demographic data** (age, gender) for better targeting

Now ask:
- What if this data is **shared with a third-party ML vendor**?
- What if the ML model **discriminates** against certain demographics?
- What if the browsing history **reveals sensitive health or political interests**?
- What if the data is **transferred to servers in another country** with weaker privacy laws?

A PIA forces you to answer these questions **before** writing any code. It's much cheaper to change a design on paper than to refactor a shipped feature.

## 📋 Step 1: Define the PIA Framework

A PIA evaluates risk across several dimensions. Let's define our scoring framework.

In [ ]:
# The PIA risk framework — each dimension contributes to the overall risk score

RISK_DIMENSIONS = {
    "data_sensitivity": {
        "description": "How sensitive is the data being collected?",
        "scoring": {
            1: "Only public data (product names, public pages)",
            2: "Internal data (usage counts, aggregated metrics)",
            3: "Confidential PII (names, emails, phone numbers)",
            4: "Restricted PII (SSN, financial, health data)",
            5: "Special categories (biometric, genetic, children's data)"
        }
    },
    "data_volume": {
        "description": "How many people are affected?",
        "scoring": {
            1: "< 100 users (internal testing)",
            2: "100–10,000 users (limited release)",
            3: "10,000–1M users (regional launch)",
            4: "1M–100M users (major product)",
            5: "> 100M users (global platform)"
        }
    },
    "third_party_sharing": {
        "description": "Is data shared outside the company?",
        "scoring": {
            1: "No sharing — data stays in our systems",
            2: "Shared with processors under contract (cloud hosting)",
            3: "Shared with partners for joint features",
            4: "Shared with advertisers or data brokers",
            5: "Made publicly available or sold"
        }
    },
    "cross_border_transfer": {
        "description": "Does data cross country borders?",
        "scoring": {
            1: "No — stays in one country",
            2: "Within same legal region (EU-to-EU, US states)",
            3: "Between adequate countries (EU-to-Canada)",
            4: "To countries without adequacy (EU-to-US without framework)",
            5: "To countries with poor privacy protections"
        }
    },
    "automated_decisions": {
        "description": "Are automated decisions made about people?",
        "scoring": {
            1: "No automated decisions",
            2: "Recommendations (user can ignore)",
            3: "Content filtering or ranking",
            4: "Eligibility decisions (credit, insurance, hiring)",
            5: "Autonomous actions with real-world impact"
        }
    },
    "retention_period": {
        "description": "How long is data kept?",
        "scoring": {
            1: "Not stored — processed and discarded",
            2: "< 30 days (session data, temporary)",
            3: "30 days – 1 year (operational use)",
            4: "1–7 years (legal requirements)",
            5: "Indefinite or no defined retention"
        }
    }
}

# Print the framework so users can reference it
print("📋 PIA Risk Scoring Framework")
print("=" * 70)
for dim_name, dim in RISK_DIMENSIONS.items():
    print(f"\n📊 {dim_name.replace('_', ' ').title()}")
    print(f"   {dim['description']}")
    for score, desc in dim["scoring"].items():
        print(f"   {score} — {desc}")

## 🏗️ Step 2: Build the PIA Workflow Engine

Now let's build the actual PIA system. It will:
1. Accept a feature description and data flow
2. Score each risk dimension
3. Calculate an overall risk level
4. Determine what approvals are needed
5. Store the PIA for tracking

In [ ]:
class PrivacyImpactAssessment:
    """A complete PIA workflow engine."""

    # Overall risk bands, keyed by the upper bound of the adjusted score.
    RISK_BANDS = [
        (1.5, "negligible"),
        (2.5, "low"),
        (3.5, "medium"),
        (4.5, "high"),
        (float("inf"), "critical"),
    ]

    LEVEL_INFO = {
        "negligible": {"approval": "self-approve",    "color": "🟢", "score": 1},
        "low":        {"approval": "team lead",       "color": "🔵", "score": 2},
        "medium":     {"approval": "privacy team",    "color": "🟡", "score": 3},
        "high":       {"approval": "privacy + legal", "color": "🟠", "score": 4},
        "critical":   {"approval": "CISO + DPO",      "color": "🔴", "score": 5},
    }

    # GDPR Art. 35 makes a DPIA mandatory for special categories of data and for
    # systematic automated decision-making. A feature that maxes out either of
    # those dimensions cannot be talked down by five comfortable ones, so those
    # two dimensions set a floor on the overall level.
    HIGH_RISK_DIMENSIONS = ("data_sensitivity", "automated_decisions")

    # Cap on how much credit mitigations can claim, in score points. Without a
    # cap a team lists ten paper mitigations and walks a Critical feature down
    # to "self-approve" without changing a line of the design.
    MAX_MITIGATION_CREDIT = 0.5

    def __init__(self, feature_name, team, assessor):
        self.feature_name = feature_name
        self.team = team
        self.assessor = assessor
        self.description = ""
        self.data_flows = []
        self.scores = {}
        self.mitigations = []
        self.status = "draft"
        self.created_at = datetime.now()

    def set_description(self, description):
        """Describe what the feature does."""
        self.description = description

    def add_data_flow(self, source, destination, data_types, purpose):
        """Document a data flow: where data comes from and where it goes."""
        self.data_flows.append({
            "source": source,
            "destination": destination,
            "data_types": data_types,
            "purpose": purpose
        })

    def score_dimension(self, dimension, score, justification=""):
        """Score a risk dimension (1-5)."""
        if dimension not in RISK_DIMENSIONS:
            raise ValueError(f"Unknown dimension: {dimension}")
        if score < 1 or score > 5:
            raise ValueError("Score must be 1-5")
        self.scores[dimension] = {
            "score": score,
            "justification": justification
        }

    def add_mitigation(self, risk, mitigation, reduces_score_by=0):
        """Add a mitigation measure for an identified risk."""
        self.mitigations.append({
            "risk": risk,
            "mitigation": mitigation,
            "score_reduction": reduces_score_by
        })

    def _band_for(self, score):
        """The risk level a numeric score falls into."""
        for upper, level in self.RISK_BANDS:
            if score < upper:
                return level
        return "critical"

    def calculate_risk(self):
        """Calculate the overall risk level and the approval it requires.

        Two rules matter more than the arithmetic:

        1. **Mitigation credit is capped.** Mitigations reduce the overall score
           directly (that is what the report says they do), but by at most
           MAX_MITIGATION_CREDIT in total, however many are listed.
        2. **The mean is a summary, not a judgement.** Averaging six dimensions
           lets one Critical dimension hide behind five quiet ones: AI credit
           scoring scores 5 on data sensitivity and 5 on automated decisions and
           still averages to 3.8, which is merely "high". So a maxed-out
           dimension sets a floor the average cannot undercut, and mitigations
           cannot push the level below that floor. You do not mitigate your way
           out of a Critical dimension on paper — you change the design.
        """
        if not self.scores:
            return None

        avg_score = sum(s["score"] for s in self.scores.values()) / len(self.scores)

        claimed_credit = sum(m["score_reduction"] for m in self.mitigations)
        applied_credit = min(claimed_credit, self.MAX_MITIGATION_CREDIT)
        adjusted_score = max(1.0, avg_score - applied_credit)

        level = self._band_for(adjusted_score)

        # Escalation floor from any dimension scored 5.
        maxed = {dim for dim, s in self.scores.items() if s["score"] == 5}
        floor = None
        if maxed:
            floor = "medium"
        if maxed & set(self.HIGH_RISK_DIMENSIONS):
            floor = "high"
        if set(self.HIGH_RISK_DIMENSIONS) <= maxed:
            floor = "critical"

        escalated_from = None
        if floor and self.LEVEL_INFO[floor]["score"] > self.LEVEL_INFO[level]["score"]:
            escalated_from, level = level, floor

        info = self.LEVEL_INFO[level]
        return {
            "raw_score": round(avg_score, 2),
            "adjusted_score": round(adjusted_score, 2),
            "claimed_credit": round(claimed_credit, 2),
            "applied_credit": round(applied_credit, 2),
            "level": level,
            "band_score": info["score"],
            "approval_required": info["approval"],
            "color": info["color"],
            "escalated_from": escalated_from,
            "maxed_dimensions": sorted(maxed),
        }

    def generate_report(self):
        """Generate a human-readable PIA report."""
        risk = self.calculate_risk()

        report = []
        report.append("=" * 70)
        report.append("PRIVACY IMPACT ASSESSMENT REPORT")
        report.append("=" * 70)
        report.append(f"Feature:    {self.feature_name}")
        report.append(f"Team:       {self.team}")
        report.append(f"Assessor:   {self.assessor}")
        report.append(f"Date:       {self.created_at.strftime('%Y-%m-%d')}")
        report.append(f"Status:     {self.status.upper()}")
        report.append("")
        report.append(f"Description: {self.description}")

        # Data flows
        report.append("\n" + "-" * 70)
        report.append("DATA FLOWS")
        report.append("-" * 70)
        for i, flow in enumerate(self.data_flows, 1):
            report.append(f"  Flow {i}: {flow['source']} → {flow['destination']}")
            report.append(f"    Data:    {', '.join(flow['data_types'])}")
            report.append(f"    Purpose: {flow['purpose']}")

        # Risk scores
        report.append("\n" + "-" * 70)
        report.append("RISK SCORES")
        report.append("-" * 70)
        for dim, info in self.scores.items():
            bar = "█" * info["score"] + "░" * (5 - info["score"])
            report.append(f"  {dim.replace('_', ' ').title():<28} [{bar}] {info['score']}/5")
            if info["justification"]:
                report.append(f"    ↳ {info['justification']}")

        # Mitigations
        if self.mitigations:
            report.append("\n" + "-" * 70)
            report.append("MITIGATIONS")
            report.append("-" * 70)
            for m in self.mitigations:
                report.append(f"  🛡️ Risk: {m['risk']}")
                report.append(f"     Fix:  {m['mitigation']}")
                if m["score_reduction"]:
                    report.append(f"     Claimed reduction: {m['score_reduction']} "
                                  f"(credit is capped — see overall assessment)")

        # Overall risk
        if risk:
            report.append("\n" + "=" * 70)
            report.append("OVERALL RISK ASSESSMENT")
            report.append("=" * 70)
            report.append(f"  {risk['color']} Risk Level:      {risk['level'].upper()}")
            report.append(f"     Raw Score:       {risk['raw_score']}/5.0 (mean of dimensions)")
            report.append(f"     Mitigations:     −{risk['applied_credit']} applied "
                          f"of −{risk['claimed_credit']} claimed "
                          f"(cap −{self.MAX_MITIGATION_CREDIT})")
            report.append(f"     Adjusted Score:  {risk['adjusted_score']}/5.0")
            if risk["escalated_from"]:
                report.append(
                    f"     ⚠️ ESCALATED:    {risk['escalated_from']} → {risk['level']} "
                    f"because {', '.join(risk['maxed_dimensions'])} scored 5/5"
                )
            report.append(f"     Registry Band:   {risk['band_score']}/5 "
                          f"({risk['level']})")
            report.append(f"     Approval Needed: {risk['approval_required']}")

        return "\n".join(report)

print("✅ PIA engine loaded")

# The scoring rules this notebook claims, checked against the framework.
_low = PrivacyImpactAssessment("check-low", "t", "t")
for _dim in RISK_DIMENSIONS:
    _low.score_dimension(_dim, 1)
assert _low.calculate_risk()["level"] == "negligible"

_credit = PrivacyImpactAssessment("check-escalation", "t", "t")
for _dim, _s in [("data_sensitivity", 5), ("data_volume", 4), ("third_party_sharing", 2),
                 ("cross_border_transfer", 3), ("automated_decisions", 5),
                 ("retention_period", 4)]:
    _credit.score_dimension(_dim, _s)
_risk = _credit.calculate_risk()
assert _risk["raw_score"] == 3.83, f"expected mean 3.83, got {_risk['raw_score']}"
assert _risk["level"] == "critical", (
    "a feature scoring 5 on both data sensitivity and automated decisions must "
    "come out CRITICAL — the plain mean says 3.83 (high), which is exactly the "
    "failure mode the escalation floor exists to prevent"
)

# ...and mitigations must not be able to talk it back down.
for _i in range(10):
    _credit.add_mitigation(f"paper risk {_i}", "a promise", reduces_score_by=1.0)
assert _credit.calculate_risk()["level"] == "critical", (
    "ten claimed mitigations must not lower a Critical feature"
)
assert _credit.calculate_risk()["applied_credit"] == 0.5, "mitigation credit must be capped"
print("✅ Risk-scoring assertions passed (escalation floor + mitigation cap)")

## 📝 Step 3: Create a PIA for a Real Feature

Let's walk through a complete PIA for a realistic feature: **"Personalized Product Recommendations"**.

This is the type of feature that would trigger a PIA at Microsoft, Google, or Amazon because it:
- Collects behavioral data (browsing, purchases)
- Uses automated decision-making (ML model)
- May involve a third-party ML service
- Processes data from millions of users

In [ ]:
# Create the PIA
pia = PrivacyImpactAssessment(
    feature_name="Personalized Product Recommendations",
    team="Product Engineering",
    assessor="privacy-lab-student"
)

# Describe the feature
pia.set_description(
    "ML-powered recommendation engine that suggests products based on "
    "browsing history, purchase history, and user demographics. "
    "Recommendations appear on the homepage and product pages."
)

# Document data flows — where does data move?
pia.add_data_flow(
    source="User Browser",
    destination="Activity Log (PostgreSQL)",
    data_types=["page views", "click events", "search queries", "IP address"],
    purpose="Track user behavior to train recommendation model"
)

pia.add_data_flow(
    source="Activity Log (PostgreSQL)",
    destination="ML Training Pipeline (internal)",
    data_types=["anonymized user IDs", "product interactions", "timestamps"],
    purpose="Train recommendation model on aggregated behavior patterns"
)

pia.add_data_flow(
    source="ML Model",
    destination="User Browser (via API)",
    data_types=["recommended product IDs", "relevance scores"],
    purpose="Display personalized recommendations to the user"
)

pia.add_data_flow(
    source="User Profile (PostgreSQL)",
    destination="ML Training Pipeline",
    data_types=["age range", "city", "signup date"],
    purpose="Demographic features for model training (no names or emails)"
)

print("✅ Feature description and data flows documented")
print(f"   {len(pia.data_flows)} data flows registered")

In [ ]:
# Score each risk dimension with justification

pia.score_dimension(
    "data_sensitivity", 3,
    "Collects browsing behavior and demographics — Confidential PII"
)

pia.score_dimension(
    "data_volume", 4,
    "Feature targets all users — estimated 5M active users"
)

pia.score_dimension(
    "third_party_sharing", 1,
    "ML model runs internally — no data leaves our infrastructure"
)

pia.score_dimension(
    "cross_border_transfer", 2,
    "ML training runs in US-West, users are US-only for now"
)

pia.score_dimension(
    "automated_decisions", 2,
    "Recommendations only — user can ignore them, no access is restricted"
)

pia.score_dimension(
    "retention_period", 3,
    "Activity logs kept 90 days, model retrained monthly"
)

# Add mitigations to reduce risk
pia.add_mitigation(
    risk="Browsing data reveals sensitive interests (health, politics)",
    mitigation="Exclude health and political categories from recommendation features",
    reduces_score_by=0.5
)

pia.add_mitigation(
    risk="Model could discriminate by demographics",
    mitigation="Run fairness audit on model outputs before deployment",
    reduces_score_by=0.3
)

pia.add_mitigation(
    risk="Users cannot control what data is used",
    mitigation="Add opt-out toggle in user privacy settings",
    reduces_score_by=0.2
)

# Generate and print the full report
print(pia.generate_report())

## 💾 Step 4: Save the PIA to the Database

PIAs need to be stored for compliance — regulators may ask to see them years later. Let's save our PIA to PostgreSQL and cache the status in Redis.

In [ ]:
def save_pia(pia):
    """Save a PIA to the database."""
    risk = pia.calculate_risk()
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO privacy_impact_assessments
            (feature_name, description, team, assessor,
             data_types_collected, purpose, third_party_sharing,
             cross_border_transfer, automated_decision_making,
             risk_score, status, expires_at)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        RETURNING id
    """, (
        pia.feature_name,
        pia.description,
        pia.team,
        pia.assessor,
        json.dumps([f["data_types"] for f in pia.data_flows]),
        json.dumps([f["purpose"] for f in pia.data_flows]),
        pia.scores.get("third_party_sharing", {}).get("score", 1) > 2,
        pia.scores.get("cross_border_transfer", {}).get("score", 1) > 2,
        pia.scores.get("automated_decisions", {}).get("score", 1) > 2,
        # `risk_score` is an INTEGER 1-5 column: the band from the README's risk
        # table, not the raw mean. Handing PostgreSQL 3.83 here would silently
        # round it to 4 and the dashboard would disagree with the report — and
        # worse, a Critical feature escalated up from a 3.83 mean would be filed
        # as a 4. Store the band that matches the level we assigned.
        risk["band_score"] if risk else None,
        "in_review",
        datetime.now() + timedelta(days=365)  # PIAs expire after 1 year
    ))

    pia_id = cursor.fetchone()[0]
    conn.commit()
    conn.close()

    # Cache in Redis for quick status lookups
    r = get_redis_client()
    r.hset(f"pia:{pia.feature_name}", mapping={
        "id": str(pia_id),
        "status": "in_review",
        "risk_level": risk["level"] if risk else "unknown",
        "risk_score": str(risk["band_score"]) if risk else "0",
        "adjusted_score": str(risk["adjusted_score"]) if risk else "0",
        "approval_required": risk["approval_required"] if risk else "unknown",
        "expires_at": (datetime.now() + timedelta(days=365)).isoformat()
    })

    return pia_id

pia_id = save_pia(pia)
print(f"✅ PIA saved with ID: {pia_id}")
print(f"   Status: in_review")
print(f"   Expires: {(datetime.now() + timedelta(days=365)).strftime('%Y-%m-%d')}")

# Show it in Redis
r = get_redis_client()
cached = r.hgetall(f"pia:{pia.feature_name}")
print(f"\n🔍 Redis cache: {json.dumps(cached, indent=2)}")

# The number in the database must mean the same thing as the number in the
# report. Storing a float in an INTEGER column is a silent lie.
conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT risk_score FROM privacy_impact_assessments WHERE id = %s", (pia_id,))
stored_score = cursor.fetchone()[0]
conn.close()
_risk = pia.calculate_risk()
assert stored_score == _risk["band_score"], (
    f"stored risk_score {stored_score} != band {_risk['band_score']} for level "
    f"'{_risk['level']}' — the registry and the report disagree"
)
assert cached["risk_level"] == _risk["level"], "Redis and the PIA object disagree"
print(f"✅ Stored band {stored_score}/5 matches level '{_risk['level']}'")

## 📊 Step 5: PIA Review Dashboard

In a real company, the privacy team needs a dashboard to see all pending PIAs, their risk levels, and which ones need attention. Let's build a simple one.

In [ ]:
# First, let's create a few more PIAs so the dashboard has data to show

sample_features = [
    {
        "name": "Email Newsletter Signup",
        "team": "Marketing",
        "description": "Collect email and name for weekly newsletter",
        "scores": {"data_sensitivity": 3, "data_volume": 3, "third_party_sharing": 3,
                   "cross_border_transfer": 1, "automated_decisions": 1, "retention_period": 4}
    },
    {
        "name": "Customer Support Chat",
        "team": "Support Engineering",
        "description": "Live chat with support agents, transcripts stored",
        "scores": {"data_sensitivity": 4, "data_volume": 3, "third_party_sharing": 1,
                   "cross_border_transfer": 1, "automated_decisions": 1, "retention_period": 3}
    },
    {
        "name": "AI Credit Scoring",
        "team": "FinTech",
        "description": "ML model that scores creditworthiness using financial data",
        "scores": {"data_sensitivity": 5, "data_volume": 4, "third_party_sharing": 2,
                   "cross_border_transfer": 3, "automated_decisions": 5, "retention_period": 4}
    },
    {
        "name": "Public Product Reviews",
        "team": "Product Engineering",
        "description": "Users post public reviews with display name and rating",
        "scores": {"data_sensitivity": 1, "data_volume": 3, "third_party_sharing": 1,
                   "cross_border_transfer": 1, "automated_decisions": 1, "retention_period": 5}
    },
]

sample_pias = {}
for feature in sample_features:
    p = PrivacyImpactAssessment(feature["name"], feature["team"], "auto-assessment")
    p.set_description(feature["description"])
    for dim, score in feature["scores"].items():
        p.score_dimension(dim, score)
    sample_pias[feature["name"]] = {"pia": p, "id": save_pia(p), "risk": p.calculate_risk()}

print(f"✅ Created {len(sample_features)} additional PIAs")
for name, entry in sample_pias.items():
    risk = entry["risk"]
    escalation = (f"  ⚠️ escalated from {risk['escalated_from']} "
                  f"({', '.join(risk['maxed_dimensions'])} = 5/5)"
                  if risk["escalated_from"] else "")
    print(f"   {risk['color']} #{entry['id']:<3} {name:<28} mean {risk['raw_score']} "
          f"→ {risk['level'].upper()}{escalation}")

# The README's risk table calls AI-based credit scoring a 5 / Critical. The
# engine has to agree with the documentation students are reading.
assert sample_pias["AI Credit Scoring"]["risk"]["level"] == "critical", (
    "AI credit scoring must land Critical — its mean is only 3.83, so this "
    "passes only because of the escalation floor"
)
assert sample_pias["Public Product Reviews"]["risk"]["escalated_from"] == "low", (
    "indefinite retention (5/5) must pull an otherwise-quiet feature up out of "
    "the 'low' band"
)

In [ ]:
def display_pia_dashboard():
    """Show all PIAs in a dashboard format."""
    conn = get_db_connection()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cursor.execute("""
        SELECT id, feature_name, team, risk_score, status,
               third_party_sharing, cross_border_transfer,
               automated_decision_making, created_at, expires_at
        FROM privacy_impact_assessments
        ORDER BY risk_score DESC NULLS LAST
    """)

    pias = cursor.fetchall()
    conn.close()

    print("📊 Privacy Impact Assessment Dashboard")
    print("=" * 90)
    print(f"  {'ID':<4} {'Feature':<35} {'Team':<20} {'Risk':<16} {'Status':<12} {'Flags'}")
    print("-" * 90)

    # risk_score is the 1-5 band, so the dashboard can show the same level name
    # and colour the report used. Two views of one number, never two numbers.
    band_to_level = {info["score"]: level
                     for level, info in PrivacyImpactAssessment.LEVEL_INFO.items()}

    for p in pias:
        band = int(p["risk_score"]) if p["risk_score"] else 0
        level = band_to_level.get(band, "unscored")
        icon = PrivacyImpactAssessment.LEVEL_INFO.get(level, {}).get("color", "⚪")

        # Risk flags
        flags = []
        if p["third_party_sharing"]:
            flags.append("3rd-party")
        if p["cross_border_transfer"]:
            flags.append("cross-border")
        if p["automated_decision_making"]:
            flags.append("auto-decision")

        print(f"  {p['id']:<4} {p['feature_name'][:34]:<35} {p['team'][:19]:<20} "
              f"{icon}{band}/5 {level:<11} {p['status']:<12} "
              f"{', '.join(flags) if flags else '—'}")

    # Summary
    print("\n" + "=" * 90)
    print(f"  Total PIAs: {len(pias)}")
    print(f"  ⚠️  High/Critical risk: "
          f"{sum(1 for p in pias if p['risk_score'] and int(p['risk_score']) >= 4)}")
    print(f"  ⏰ Expiring in 30 days: {sum(1 for p in pias if p['expires_at'] and p['expires_at'] < datetime.now() + timedelta(days=30))}")

display_pia_dashboard()

## 🔄 Step 6: PIA Lifecycle Management

PIAs aren't one-and-done. They have a lifecycle:

```
draft → in_review → approved → [expires after 1 year] → renewal needed
                  ↘ rejected → revised → in_review (again)
```

Let's implement approval and expiry checking.

In [ ]:
def approve_pia(pia_id, approver):
    """Approve a PIA — changes status and records who approved it."""
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        UPDATE privacy_impact_assessments
        SET status = 'approved', approved_by = %s
        WHERE id = %s AND status = 'in_review'
        RETURNING feature_name, risk_score
    """, (approver, pia_id))

    result = cursor.fetchone()
    conn.commit()
    conn.close()

    if result:
        # Update Redis cache
        r = get_redis_client()
        r.hset(f"pia:{result[0]}", "status", "approved")
        print(f"✅ PIA #{pia_id} '{result[0]}' approved by {approver}")
    else:
        print(f"❌ PIA #{pia_id} not found or not in review")

def reject_pia(pia_id, reason):
    """Reject a PIA — feature cannot ship until issues are fixed."""
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        UPDATE privacy_impact_assessments
        SET status = 'rejected'
        WHERE id = %s AND status = 'in_review'
        RETURNING feature_name
    """, (pia_id,))

    result = cursor.fetchone()
    conn.commit()
    conn.close()

    if result:
        r = get_redis_client()
        r.hset(f"pia:{result[0]}", "status", "rejected")
        print(f"❌ PIA #{pia_id} '{result[0]}' rejected")
        print(f"   Reason: {reason}")

def find_pia_id(feature_name):
    """Most recent PIA id for a feature.

    Never hardcode the ids: re-running the cells above inserts new rows, and a
    gate that approves "PIA #1" starts approving somebody else's feature the
    second the table changes underneath it.
    """
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id FROM privacy_impact_assessments
        WHERE feature_name = %s
        ORDER BY id DESC LIMIT 1
    """, (feature_name,))
    row = cursor.fetchone()
    conn.close()
    return row[0] if row else None


def check_pia_before_deploy(feature_name):
    """Check if a feature has an approved, non-expired PIA. Called in CI/CD."""
    # Try the Redis cache first...
    r = get_redis_client()
    record = r.hgetall(f"pia:{feature_name}")
    source = "cache"

    # ...but a cache miss is not evidence of absence. Redis is a cache; the
    # database is the record. If a flush could turn "rejected" into "no PIA
    # found", the difference would not matter here (both block) — but it would
    # also lose the reason, and an approved PIA would stop working for no
    # reason anyone could explain.
    if not record:
        conn = get_db_connection()
        cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
        cursor.execute("""
            SELECT status, expires_at FROM privacy_impact_assessments
            WHERE feature_name = %s
            ORDER BY id DESC LIMIT 1
        """, (feature_name,))
        row = cursor.fetchone()
        conn.close()
        if row:
            record = {"status": row["status"],
                      "expires_at": row["expires_at"].isoformat() if row["expires_at"] else ""}
            source = "database"

    if not record:
        return False, "No PIA found for this feature — create one before deploying"

    status = record.get("status", "unknown")
    if status != "approved":
        return False, f"PIA status is '{status}' ({source}) — must be 'approved' to deploy"

    expires = record.get("expires_at", "")
    if expires and datetime.fromisoformat(expires) < datetime.now():
        return False, "PIA has expired — renewal required"

    return True, f"PIA approved and valid ({source})"

# Demo: approve the low-risk PIA, reject the high-risk one
print("📝 PIA Review Decisions\n")

# Approve the recommendation engine PIA (we added mitigations)
approve_pia(find_pia_id("Personalized Product Recommendations"), "privacy-officer@company.com")

# Reject the AI credit scoring PIA (Critical — mitigations on paper are not enough)
reject_pia(find_pia_id("AI Credit Scoring"),
           "Automated credit decisions require human-in-the-loop override per EU AI Act")

# Check if features can deploy
print("\n🚀 Deployment Gate Checks\n")
gate_results = {}
for feature in ["Personalized Product Recommendations", "AI Credit Scoring", "Unregistered Feature"]:
    can_deploy, reason = check_pia_before_deploy(feature)
    gate_results[feature] = can_deploy
    icon = "✅" if can_deploy else "🚫"
    print(f"  {icon} {feature}")
    print(f"     {reason}")

# The gate is the only thing standing between a rejected PIA and production.
assert gate_results["Personalized Product Recommendations"] is True, "approved PIA must deploy"
assert gate_results["AI Credit Scoring"] is False, "a rejected PIA must block deployment"
assert gate_results["Unregistered Feature"] is False, "no PIA must fail closed"

# And it must survive losing the cache — this is a compliance control, not a
# performance optimisation.
get_redis_client().delete("pia:Personalized Product Recommendations")
can_deploy, reason = check_pia_before_deploy("Personalized Product Recommendations")
assert can_deploy is True, f"gate broke when the cache was flushed: {reason}"
print(f"\n✅ Deployment-gate assertions passed (survived a cache flush: {reason})")

## 🎯 Key Takeaways

1. **PIAs are mandatory gates** — no feature ships without one at companies like Microsoft
2. **Risk scoring is structured** — use consistent dimensions so PIAs are comparable
3. **Data flows are documented** — know exactly where personal data moves
4. **Mitigations reduce risk** — encryption, opt-outs, and access controls lower the score
5. **PIAs expire** — privacy landscapes change, so reassess annually
6. **Automate the gate** — check PIA status in CI/CD so features can't bypass review, and read it from the record, not only from a cache
7. **Never average away a Critical dimension** — the mean of six dimensions puts AI credit scoring at 3.8 ("high"). It is a 5 on data sensitivity and a 5 on automated decisions, and GDPR Art. 35 makes a DPIA mandatory for either. A maxed-out dimension sets a floor the average cannot undercut.
8. **Cap mitigation credit** — mitigations that exist only in the PIA document are free to write. If listing them can walk a feature down a band, they will be listed.

### What this notebook does *not* do

The scoring here is a teaching scaffold, not a compliance product. A real PIA
programme also needs: a legal basis recorded per purpose (consent, contract,
legitimate interest — with the balancing test written down), a data-flow diagram
someone outside the team can follow, a DPO consultation record, retention and
deletion commitments per data element, sub-processor contracts, and a
re-assessment trigger when the feature changes rather than only when the
calendar says a year has passed. Numbers make PIAs comparable; they do not make
them correct.

### What Microsoft Does

- Every Azure service has a PIA in a central registry
- The **SDL (Security Development Lifecycle)** requires PIAs at design phase
- PIAs are linked to feature flags — rejected PIAs block deployment
- Annual PIA renewal is automated — teams get reminders 60 days before expiry
- The Chief Privacy Officer reviews all Critical-risk PIAs personally

### Next Notebook

In **Notebook 3: Anonymization Techniques**, we'll learn how to protect data using k-anonymity, l-diversity, differential privacy, and tokenization.